In [ ]:
from pdb import set_trace as st
from pprint import pprint
import json

import msgspec
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import os
import re 

from concurrent.futures import ThreadPoolExecutor
from sutime import SUTime

global sutime, heideltime_parser
sutime = SUTime(mark_time_ranges=True, include_range=True)

from python_heideltime import Heideltime

heideltime_parser = Heideltime()
heideltime_parser.set_document_type("NEWS")

from py_heideltime import heideltime

from lxml import etree
parser = etree.XMLParser()

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

timex_text_pattern = re.compile(r"<TIMEX3[^>]*>(.*?)</TIMEX3>")
timex_value_pattern = re.compile(r'<TIMEX3[^>]*\bvalue="([^"]+)"')
timex_pattern = re.compile(r'<TIMEX3[^>]*\bvalue="([^"]+)"[^>]*>(.*?)</TIMEX3>')

[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Registering annotator sutime with class edu.stanford.nlp.time.TimeAnnotator
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator tokenize
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ssplit
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator pos
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator lemma
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ner
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator sutime


# Utils

In [3]:
def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()

    with open(file_path, "rb") as file:
        data = file.read()
    if jsonl:
        output = decoder.decode_lines(data)
    else:
        output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(encoder.encode(data))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)

def read_tsv(file_path, row_names=None):
    file_path = Path(file_path)
    if file_path.is_file() and file_path.suffix == ".tsv" :
        temp = pd.read_csv(file_path, sep='\t', names=row_names)
    else:
        raise ValueError("filepath is not a file or it is not a tsv file.")
    return temp


def parse_heideltime_result(heideltime_result, flatten=False):
    # root = etree.fromstring(heideltime_result, parser=parser)

    # values = []
    # for el in root.iter():
    #     if el.tag == "TIMEX3":
    #         value = el.attrib.get("value", "")
    #         if value and "2025" not in value and not value.startswith("P"):
    #             values.append({"text": el.text or "", "value": value})

    heideltime_result = heideltime_result.split("\n")[3]
    values = timex_text_pattern.findall(heideltime_result)

    # if flatten:
    #     return [field for item in values for field in (item["text"], item["value"])]

    return values


def parse_sutime_result(sutime_result, flatten=False):
    values = [x.get("text") for x in sutime_result if x.get("text")]
    # for x in sutime_result:
    #     text = x.get("text", "")
    #     # value = x.get("value", "")

    #     if isinstance(value, dict):
    #         value = ""
    #     elif not value or ("2025" not in value and value.startswith("P")):
    #         value = ""

    #     if flatten:
    #         values.append(text)
    #         if value:
    #             values.append(value)
    #     else:
    #         values.append({"text": text, "value": value})

    return values

# Read the corpus data

In [47]:
DATA_PATH = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize")
SPLIT = "train"

CORPUS_TEMPORAL_JSONL_PATH = DATA_PATH / SPLIT / "chunked/corpus_temporal_refined.jsonl"
corpus_temporal_jsonl = read_json(CORPUS_TEMPORAL_JSONL_PATH, jsonl=True)

CHUNKED_CORPUS_JSONL_PATH = DATA_PATH / SPLIT / "chunked/corpus.jsonl"
chunked_corpus_jsonl = read_json(CHUNKED_CORPUS_JSONL_PATH, jsonl=True)

STITCHED_CORPUS_JSONL = []
for x, y in zip(chunked_corpus_jsonl, corpus_temporal_jsonl):
    x.update(y)
    STITCHED_CORPUS_JSONL.append(x)

The file is of type: <class 'list'>
The file contains 24804 items.
The file is of type: <class 'list'>
The file contains 24804 items.


In [48]:
texts = [x["text"] for x in STITCHED_CORPUS_JSONL]
llm_results = [x["temporal"] for x in STITCHED_CORPUS_JSONL]

In [ ]:
results = []

def init_worker():
    global sutime, heideltime_parser
    
    from sutime import SUTime
    sutime = SUTime(mark_time_ranges=True, include_range=True)

    from python_heideltime import Heideltime
    heideltime_parser = Heideltime()
    heideltime_parser.set_document_type("NEWS")


def get_temporal_set(text, llm_result, flatten=True):
    sutime_result = parse_sutime_result(sutime.parse(text), flatten=flatten)

    heideltime_result = parse_heideltime_result(
        heideltime_parser.parse(text), flatten=flatten
    )
    llm_result = llm_result.split(",")

    if flatten:
        return set(sutime_result).union(heideltime_result).union(llm_result)
    else:
        return sutime_result, heideltime_result, llm_result


def get_temporal_set_worker(args):
    text, llm_result = args
    return get_temporal_set(text, llm_result, flatten=True)

# if __name__ == "__main__":
#     with ThreadPoolExecutor(max_workers=16) as executor:
#         results = list(tqdm(
#             executor.map(get_temporal_set_worker, zip(texts, llm_results)),
#             total=len(texts)  # important to get progress bar length right
#         ))

In [ ]:
# write_json(str(DATA_PATH / SPLIT / "chunked/temporal_sutime_heideltime.jsonl"), results, jsonl=True)

The file contains 100 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/chunked/temporal_sutime_heideltime.jsonl


# Adding temporal expressions for each positive and negative passage in the training jsonl

In [3]:
DATA_PATH = Path(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize"
)
SPLIT = "train"

TRAIN_JSONL_PATH = DATA_PATH / SPLIT / "train.jsonl"
train_jsonl = read_json(TRAIN_JSONL_PATH, jsonl=True)

The file is of type: <class 'list'>
The file contains 8060 items.


In [ ]:
for ix, line in tqdm(enumerate(train_jsonl), total=len(train_jsonl)):
    positive_passages, negative_passages = line["positive_passages"], line["negative_passages"]

    for item in positive_passages:
        pos_docid = item["docid"]
        pos_text = item["text"]

        sutime_result = sutime.parse(pos_text)
        sutime_result = parse_sutime_result(sutime_result)

        heideltime_result = heideltime_parser.parse(pos_text)
        heideltime_result = parse_heideltime_result(heideltime_result)

        item['temporal'] = list(set(sutime_result + heideltime_result))

    for item in negative_passages:
        neg_docid = item["docid"]
        neg_text = item["text"]

        sutime_result = sutime.parse(neg_text)
        sutime_result = parse_sutime_result(sutime_result)

        heideltime_result = heideltime_parser.parse(neg_text)
        heideltime_result = parse_heideltime_result(heideltime_result)

        item["temporal"] = list(set(sutime_result + heideltime_result))

In [26]:
sutime.parse("What was the official name of Blaffer Art Museum from 2012 to 2014?")

[{'timex-value': '2012',
  'start': 54,
  'end': 58,
  'text': '2012',
  'type': 'DATE',
  'value': '2012'}]

In [24]:
text = "What was the official name of Blaffer Art Museum from 2012 to 2014?"

timexs = heideltime(text, language="English", document_type="news", dct="1939-08-31")

pprint(timexs)

[{'span': [54, 58],
  'text': '2012',
  'tid': 't1',
  'type': 'DATE',
  'value': '2012'},
 {'span': [62, 66],
  'text': '2014',
  'tid': 't2',
  'type': 'DATE',
  'value': '2014'}]


In [11]:
heideltime_result = parse_heideltime_result(
    heideltime_parser.parse(text), flatten=True
)

In [9]:
pprint(heideltime_result)

['1900', '1995']


In [ ]:
# write_json(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal.jsonl",
#     train_jsonl,
#     jsonl=True
# )

The file contains 8060 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal.jsonl


# Dev TsRetriever with temporal

In [68]:
temp = {"docid":4161,"text":"Roberto Brown played for which team from 2005 to 2006?","temporal":["2005","2006","from 2005 to 2006"]}

In [70]:
docid = temp["docid"]
text = temp["text"]
temporal = temp["temporal"]

In [3]:
from datasets import load_dataset

ds = load_dataset(
    "zeta-alpha-ai/NanoNQ",
    "qrels",
    cache_dir="/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/tevatron",
)

Generating train split: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:00<00:00, 16597.84 examples/s]


In [7]:
ds['train']

Dataset({
    features: ['query-id', 'corpus-id'],
    num_rows: 57
})

In [9]:
for i in ds['train']:
    print(i)
    break

{'query-id': 'test1618', 'corpus-id': 'doc57226'}


In [12]:
with open("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/tevatron/zeta-alpha-ai___nano_nq/qrels.txt", "w") as f:
    for i in ds['train']:
        query_id, corpus_id = i['query-id'], i['corpus-id']
        f.write(f"{query_id} 0 {corpus_id} 1\n")

In [8]:
original = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal.jsonl", jsonl=True)
v2 = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl", jsonl=True)

The file is of type: <class 'list'>
The file contains 8060 items.
The file is of type: <class 'list'>
The file contains 8060 items.


In [9]:
original[0].keys()

dict_keys(['query_id', 'query', 'positive_passages', 'negative_passages'])

In [14]:
for k, v in zip(original, v2):
    if k["query"] != v["query"]:
        v["query"] = k["query"]

In [ ]:
# write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal_v2_temp.jsonl", v2, jsonl=True)

The file contains 8060 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal_v2_temp.jsonl


In [ ]:
v2_temp = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl", jsonl=True)


count = 0 
for ix, item in enumerate(v2_temp):
    for positive_passages in item["positive_passages"]:
        for temporal in positive_passages["temporal"]:
            if temporal not in positive_passages["text"]:
                print(item["query_id"])
                print(positive_passages["text"])
                print(temporal)
# print(count)
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl", v2_temp, jsonl=True)
            

The file is of type: <class 'list'>
The file contains 8060 items.
The file contains 8060 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl


In [43]:
corpus_temporal = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/chunked/corpus_temporal_refined.jsonl", jsonl=True)
corpus = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/chunked/corpus.jsonl",jsonl=True)

The file is of type: <class 'list'>
The file contains 24804 items.
The file is of type: <class 'list'>
The file contains 24804 items.


In [46]:
count = 0
for ix, (k, v) in enumerate(zip(corpus, corpus_temporal)):
    assert k['docid'] == v['docid']
    assert k['chunkid'] == v['chunkid']
    
    corpus_text = k['text'].lower().strip()
    temporal = v['temporal'].lower().strip().split(",")
    
    for t in temporal:
        if t not in corpus_text:
            partially_match = [x for x in t.split(" ") if x in corpus_text]
            if len(partially_match) == 0:
                temporal.remove(t)
                count += 1
        v['temporal'] = temporal
print(count)

3819


In [47]:
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/chunked/corpus_temporal_refined_v2.jsonl", corpus_temporal, jsonl=True)

The file contains 24804 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/chunked/corpus_temporal_refined_v2.jsonl


# Check BGE temporal

In [231]:
model_name_or_path = "/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/ts-retriever/bge-base-en-v1.5/svd"

from sentence_transformers import models, SentenceTransformer

# Load the transformer and SentenceTransformer model
transformer_model = models.Transformer.load(model_name_or_path, trust_remote_code=True)

pooling_model = models.Pooling(
    transformer_model.get_word_embedding_dimension(), pooling_mode="cls"
)
normalize_model = models.Normalize()

# Set up the SentenceTransformer model
my_model = SentenceTransformer(
    modules=[transformer_model, pooling_model, normalize_model],
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "max_seq_length": 512,
    },
    trust_remote_code=True,
)

In [221]:
# Load model directly
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
model = AutoModel.from_pretrained("BAAI/bge-base-en-v1.5")
model.cuda()

# Compute token embeddings
def get_sentence_embeddings(model, input_list):
    input = tokenizer(input_list, padding=True, truncation=True, return_tensors='pt')
    
    model.eval()
    with torch.no_grad():
        input = {k: v.cuda() for k, v in input.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            # print(torch.get_autocast_dtype("cuda"))
            output = model(**input)
        # Perform pooling. In this case, cls pooling.
        sentence_embeddings = output[0][:, 0]
        # print(sentence_embeddings.shape)
    return sentence_embeddings
    # # normalize embeddings
    # return torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)

def get_cosine_similarity(a, b, normalize=True):
    # This assume that the embedding has been normalized
    
    if normalize:
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
    
    return torch.matmul(a, b.T)

In [228]:
prompt = "Represent this sentence for searching relevant passages: "

q_pos_text = prompt + "Jermaine Beckford played for which team from 2003 to 2004?"
qt_pos_text = prompt + "from 2003 to 2004?"

q_neg_text = prompt + "Jermaine Beckford played for which team from 1996 to 2002?"
qt_neg_text = prompt + "from 1996 to 2002"

p_text = "Beckford originally began his career in the Chelsea youth team , coming through the schoolboy ranks at the same time as Carlton Cole . Rejected by Chelsea in 2003 , he was signed up by Wealdstone , then in the Isthmian Premier League , and played as a semi-professional for three years whilst also working as a windscreen fitter for the RAC . His very impressive goal scoring record for Wealdstone attracted a lot of attention from Football League sides and reportedly more than 30 professional clubs showed an interest in the prolific striker , with many sending scouts to watch him play for Wealdstone . He had a trial with Championship side Crystal Palace , before signing for Leeds United in March 2006 for an undisclosed fee , having scored 35 goals in 40 games for Wealdstone that season ."

In [229]:
def show_similarity(model, q, p_pos, p_neg):
    q = get_sentence_embeddings(model, q if isinstance(q, list) else [q])
    p_pos = get_sentence_embeddings(model, p_pos if isinstance(p_pos, list) else [p_pos])
    p_neg = get_sentence_embeddings(model, p_neg if isinstance(p_neg, list) else [p_neg])

    print("q, p_pos", get_cosine_similarity(q, p_pos))
    print("q, p_neg", get_cosine_similarity(q, p_neg))

show_similarity(model, [p_text], [q_pos_text], [q_neg_text])
show_similarity(model, [p_text], [qt_pos_text], [qt_neg_text])

q, p_pos tensor([[0.5685]], device='cuda:0')
q, p_neg tensor([[0.5560]], device='cuda:0')
q, p_pos tensor([[0.3709]], device='cuda:0')
q, p_neg tensor([[0.4270]], device='cuda:0')


In [ ]:
def show_similarity(model, q, p_pos, p_neg, temporal=False):
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            q = model.encode(q if isinstance(q, list) else [q], convert_to_tensor=True)
            p_pos = model.encode(p_pos if isinstance(p_pos, list) else [p_pos], convert_to_tensor=True)
            p_neg = model.encode(p_neg if isinstance(p_neg, list) else [p_neg], convert_to_tensor=True)
    
    if temporal:
        print("qt, pt_pos", get_cosine_similarity(q[:, 512:], p_pos[:, 512:]))
        print("qt, pt_neg", get_cosine_similarity(q[:, 512:], p_neg[:, 512:]))
    else:
        print("q, p_pos", get_cosine_similarity(q, p_pos))
        print("q, p_neg", get_cosine_similarity(q, p_neg))

# my_model.disable_adapters()
my_model.enable_adapters()
show_similarity(my_model, p_text, q_pos_text, q_neg_text)
show_similarity(my_model, p_text, q_pos_text, q_neg_text, temporal=True)

q, p_pos tensor([[0.5266]], device='cuda:0')
q, p_neg tensor([[0.5170]], device='cuda:0')
qt, pt_pos tensor([[0.3581]], device='cuda:0')
qt, pt_neg tensor([[0.3672]], device='cuda:0')


In [233]:
def show_similarity(model, q, p_pos, p_neg, temporal=False):
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            q = model.encode(q if isinstance(q, list) else [q], convert_to_tensor=True)
            p_pos = model.encode(p_pos if isinstance(p_pos, list) else [p_pos], convert_to_tensor=True)
            p_neg = model.encode(p_neg if isinstance(p_neg, list) else [p_neg], convert_to_tensor=True)
    
    if temporal:
        print("qt, pt_pos", get_cosine_similarity(q[:, 512:], p_pos[:, 512:]))
        print("qt, pt_neg", get_cosine_similarity(q[:, 512:], p_neg[:, 512:]))
    else:
        print("q, p_pos", get_cosine_similarity(q, p_pos))
        print("q, p_neg", get_cosine_similarity(q, p_neg))

# my_model.disable_adapters()
my_model.enable_adapters()
show_similarity(my_model, p_text, q_pos_text, q_neg_text)
show_similarity(my_model, p_text, q_pos_text, q_neg_text, temporal=True)

q, p_pos tensor([[0.5756]], device='cuda:0')
q, p_neg tensor([[0.5682]], device='cuda:0')
qt, pt_pos tensor([[0.5368]], device='cuda:0')
qt, pt_neg tensor([[0.5114]], device='cuda:0')
